In [1]:
import os
for d, _, fs in os.walk('/kaggle/input'):
    for f in fs: print(os.path.join(d, f))

/kaggle/input/datasets/aminexdr/bhc-mimic-iv-summary/BHC_MIMIC-IV.csv
/kaggle/input/notebooks/kaziushno/thesis-a-generation/truncation_report.csv
/kaggle/input/notebooks/kaziushno/thesis-a-generation/experiment_config.json
/kaggle/input/notebooks/kaziushno/thesis-a-generation/__results__.html
/kaggle/input/notebooks/kaziushno/thesis-a-generation/selected_cases.csv
/kaggle/input/notebooks/kaziushno/thesis-a-generation/generated_summaries.csv
/kaggle/input/notebooks/kaziushno/thesis-a-generation/__huggingface_repos__.json
/kaggle/input/notebooks/kaziushno/thesis-a-generation/__notebook__.ipynb
/kaggle/input/notebooks/kaziushno/thesis-a-generation/__output__.json
/kaggle/input/notebooks/kaziushno/thesis-a-generation/gen_checkpoint.jsonl
/kaggle/input/notebooks/kaziushno/thesis-a-generation/custom.css


In [2]:
from huggingface_hub import HfApi
print(HfApi().dataset_info("mteb/summeval").sha)

bfc121155064afa2d81b5505682ffc0d96f4334c


In [ ]:
"""
================================================================================
 Evaluating Automatic Metrics and LLM-as-Judge for Clinical Text Summarization
 Final reproducible pipeline  --  single file, six stages
================================================================================

HOW TO RUN ON KAGGLE
--------------------
Set STAGE below, then Save & Run All. Run stages in order, one session each.
Each stage reads the previous stage's outputs from /kaggle/working and
checkpoints every row, so a killed session resumes where it stopped.

  1  sampling + generation      GPU   ~7-8 h   <- longest
  2  automatic metrics          GPU   ~20 min
  3  judges on clinical data    GPU   ~2.5 h
  4  SummEval validation        GPU   ~2.5 h   (needs internet ON)
  5  analysis + all outputs     CPU   ~15 min
  6  final integrity check      CPU   ~1 min

Total is well over Kaggle's 12 h session cap, which is why it is split into
three sessions: {1}, then {2, 3, 4}, then {5, 6}.

CARRYING FILES BETWEEN SESSIONS
-------------------------------
Right panel -> Add Input -> Notebook Output -> pick the previous stage's run.
Then set PREV_INPUT to the mounted path. Files are copied into /kaggle/working
at startup so every stage writes to one place.
================================================================================
"""

# =============================================================================
# STAGE SELECTOR  -- edit this line, nothing else
# =============================================================================
#   Session A: STAGES = {1}
#   Session B: STAGES = {2, 3, 4}
#   Session C: STAGES = {5, 6}
# A set is used rather than a single value so a session can run several stages
# without "all" dragging generation back in.
STAGES = {2}

# Path to the previous stage's outputs, or None on the first stage.
PREV_INPUT = "/kaggle/input/notebooks/kaziushno/thesis-a-generation"
# e.g. PREV_INPUT = "/kaggle/input/notebooks/<user>/<notebook>"

# =============================================================================
# 0. CONFIG
# =============================================================================
import os, sys, re, gc, json, time, random, hashlib, platform, shutil, subprocess
from dataclasses import dataclass, asdict, field

import numpy as np
import pandas as pd

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./work")
OUTPUTS = os.path.join(WORK, "outputs")
os.makedirs(WORK, exist_ok=True)
os.makedirs(OUTPUTS, exist_ok=True)


@dataclass(frozen=True)
class ExperimentConfig:
    seed: int = 42

    # ---- data ----
    data_path: str = "/kaggle/input/datasets/aminexdr/bhc-mimic-iv-summary/BHC_MIMIC-IV.csv"
    col_input: str = "input"
    col_ref: str = "target"
    demonstration_count: int = 2
    evaluation_count: int = 300

    # ---- models ----
    gen_model: str = "Qwen/Qwen2.5-7B-Instruct"
    weak_model: str = "facebook/bart-large-cnn"
    judge_primary: str = "Qwen/Qwen2.5-14B-Instruct"
    judge_sensitivity: str = "Qwen/Qwen2.5-7B-Instruct"

    # ---- generation limits ----
    qwen_input_limit: int = 4000
    bart_input_limit: int = 1024
    demonstration_token_limit: int = 600
    qwen_max_new_tokens: int = 512
    bart_max_new_tokens: int = 200

    # ---- judging ----
    judge_field_limit: int = 1000
    judge_max_new_tokens: int = 48      # raised from 30: 30 can clip the 3rd label

    # ---- SummEval ----
    # 50 articles x 16 system summaries = 800. Whole articles are required for the
    # summary-level statistic; do not shrink this to make a prose sentence true.
    summeval_articles: int = 50
    summeval_summaries_per_article: int = 16
    summeval_seed: int = 42
    summeval_dataset: str = "mteb/summeval"
    summeval_revision: str = "bfc121155064afa2d81b5505682ffc0d96f4334c"     # replace with a commit SHA once resolved

    # ---- BERTScore, pinned explicitly rather than left to lang='en' defaults ----
    bertscore_model_type: str = "roberta-large"
    bertscore_num_layers: int = 17
    bertscore_lang: str = "en"

    # ---- bookkeeping ----
    run_label: str = "final"

    # ---- inference ----
    bootstrap_replicates: int = 2000
    high_quantile: float = 0.75
    divergence_cutoff: float = 3.0

    conditions: tuple = ("weak_bart", "zero_shot", "few_shot")
    metrics: tuple = ("rouge1", "rouge2", "rougeL", "bleu", "bertscore_f1")


CFG = ExperimentConfig()
CONDITIONS = list(CFG.conditions)
METRICS = list(CFG.metrics)
DIMS3 = ("consistency", "completeness", "coherence")

# Fields are fingerprinted in two groups. Generation depends only on GEN_KEYS, so
# later changes to the rubric or the SummEval sample do not invalidate summaries
# that have already been produced. Splitting them this way is what lets a judging
# fix land without discarding an eight-hour generation run.
GEN_KEYS = ("seed", "data_path", "col_input", "col_ref", "demonstration_count",
            "evaluation_count", "gen_model", "weak_model", "qwen_input_limit",
            "bart_input_limit", "demonstration_token_limit",
            "qwen_max_new_tokens", "bart_max_new_tokens")
JUDGE_KEYS = ("judge_primary", "judge_sensitivity", "judge_field_limit",
              "judge_max_new_tokens")


def fingerprint(keys, extra=""):
    d = asdict(CFG)
    payload = json.dumps({k: d[k] for k in keys}, sort_keys=True, default=list) + extra
    return hashlib.sha256(payload.encode()).hexdigest()[:12]

# ---- file names (one place, used by every stage) ----
F_SELECTED   = os.path.join(WORK, "selected_cases.csv")
F_GEN        = os.path.join(WORK, "generated_summaries.csv")
F_TRUNC      = os.path.join(WORK, "truncation_report.csv")
F_METRICS    = os.path.join(WORK, "metrics_long.csv")
F_JUDGE_RAW  = os.path.join(WORK, "judge_raw_clinical.jsonl")
F_MASTER     = os.path.join(WORK, "clinical_results_final.csv")
F_SUMM_RAW   = os.path.join(WORK, "judge_raw_summeval.jsonl")
F_SUMMEVAL   = os.path.join(WORK, "summeval_final.csv")
F_CONFIG     = os.path.join(WORK, "experiment_config.json")
F_ENV        = os.path.join(WORK, "environment-final.json")
F_REQS       = os.path.join(WORK, "requirements-final.txt")
F_MANIFEST   = os.path.join(WORK, "MANIFEST.txt")
F_PUBLIC     = os.path.join(WORK, "public_aggregate_results.csv")


def stage_runs(n):
    return n in STAGES


def banner(txt):
    print("\n" + "=" * 78 + f"\n{txt}\n" + "=" * 78, flush=True)


def seed_everything(seed=CFG.seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def import_previous_outputs():
    """Copy the previous stage's files into WORK so every stage writes one place."""
    if not PREV_INPUT:
        return
    if not os.path.isdir(PREV_INPUT):
        print(f"WARNING: PREV_INPUT does not exist: {PREV_INPUT}")
        return
    allow = {"selected_cases.csv", "generated_summaries.csv", "truncation_report.csv",
             "metrics_long.csv", "judge_raw_clinical.jsonl", "clinical_results_final.csv",
             "judge_raw_summeval.jsonl", "summeval_final.csv", "gen_checkpoint.jsonl",
             "experiment_config.json", "model_revisions.json", "provenance.json",
             "judge_audit_sheet.csv"}
    n, skipped = 0, []
    for root, _, files in os.walk(PREV_INPUT):
        for fn in files:
            if fn not in allow:
                if fn.endswith((".csv", ".jsonl", ".json")):
                    skipped.append(fn)
                continue
            dst = os.path.join(WORK, fn)
            if os.path.abspath(os.path.join(root, fn)) != os.path.abspath(dst):
                shutil.copy(os.path.join(root, fn), dst)
                n += 1
    print(f"imported {n} expected file(s) from {PREV_INPUT}")
    if skipped:
        print(f"  ignored {len(skipped)} unrecognised file(s): {sorted(set(skipped))[:8]}")
    if n == 0:
        raise RuntimeError(
            f"PREV_INPUT contains none of the expected artifacts: {PREV_INPUT}\n"
            "Check the mount path with: for d,_,fs in os.walk('/kaggle/input'): print(d, fs)")


def require(path, stage_name):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{os.path.basename(path)} not found in {WORK}.\n"
            f"Run {stage_name} first, then set PREV_INPUT to its notebook output."
        )


if 4 in STAGES and CFG.summeval_revision in ("main", "", None):
    raise SystemExit(
        "\nSummEval revision is not pinned.\n"
        "Run this in a scratch cell, then paste the SHA into CFG.summeval_revision:\n"
        "    from huggingface_hub import HfApi\n"
        "    print(HfApi().dataset_info('mteb/summeval').sha)\n"
        "Failing now rather than after three hours of judging.")

seed_everything()
import_previous_outputs()
with open(F_CONFIG, "w") as f:
    json.dump(asdict(CFG), f, indent=2, default=list)


# =============================================================================
# SHARED: judge output parser
# =============================================================================
_DIM_PAT = {
    "consistency":  re.compile(r"consisten\w*\s*[:=\-]?\s*(?<!\d)([1-5])(?!\d)", re.I),
    "completeness": re.compile(r"complete\w*\s*[:=\-]?\s*(?<!\d)([1-5])(?!\d)", re.I),
    "coherence":    re.compile(r"coheren\w*\s*[:=\-]?\s*(?<!\d)([1-5])(?!\d)", re.I),
}
_BARE = re.compile(r"(?<!\d)([1-5])(?!\d)")


def parse_judge_output(text):
    """(consistency, completeness, coherence, mean, status).

    Pass 1 reads each dimension from its own label, so trailing prose or a
    '4/5' style answer cannot shift the scores.
    Pass 2 accepts exactly three bare 1-5 integers in rubric order.
    Everything else fails with a recorded reason. The parser never guesses.
    """
    if not isinstance(text, str) or not text.strip():
        return (np.nan,) * 4 + ("empty_output",)
    hits = {k: p.findall(text) for k, p in _DIM_PAT.items()}
    if all(hits.values()):
        s = [int(hits[d][0]) for d in DIMS3]
        # A label appearing twice means the model restated or revised itself; the
        # first hit is then a guess, so it is recorded rather than trusted silently.
        multi = [d for d in DIMS3 if len(hits[d]) > 1]
        status = "ok_labelled" if not multi else "ok_labelled_multi_" + "_".join(multi)
        return s[0], s[1], s[2], float(np.mean(s)), status
    bare = _BARE.findall(text)
    if len(bare) == 3:
        s = [int(x) for x in bare]
        return s[0], s[1], s[2], float(np.mean(s)), "ok_positional"
    return (np.nan,) * 4 + (f"unparsed_{len(bare)}_bare_digits",)


def _self_test_parser():
    assert parse_judge_output("CONSISTENCY: 4\nCOMPLETENESS: 3\nCOHERENCE: 5")[:4] == (4, 3, 5, 4.0)
    assert parse_judge_output("REFERENCE CONSISTENCY: 4\nCOMPLETENESS: 3\nCOHERENCE: 5")[3] == 4.0
    assert parse_judge_output("Consistency = 2, Completeness = 2, Coherence = 3")[4] == "ok_labelled"
    assert parse_judge_output(
        "CONSISTENCY: 4\nCOMPLETENESS: 3\nCOHERENCE: 5\nrevised CONSISTENCY: 2"
    )[4].startswith("ok_labelled_multi"), "repeated label must be flagged"
    assert parse_judge_output("3 4 5")[:4] == (3, 4, 5, 4.0)
    assert parse_judge_output("[3, 4, 5]")[4] == "ok_positional"
    assert np.isnan(parse_judge_output("10 4 5")[3]), "must not read 1 out of 10"
    assert np.isnan(parse_judge_output("4 4 4 5")[3]), "must reject four bare digits"
    assert np.isnan(parse_judge_output("")[3])
    assert np.isnan(parse_judge_output(None)[3])
    # the failure mode of the lenient [1-5] parser used in the pilot
    assert parse_judge_output("CONSISTENCY: 4/5\nCOMPLETENESS: 3/5\nCOHERENCE: 4/5")[:3] == (4, 3, 4)
    print("parser self-test: passed")


_self_test_parser()


# =============================================================================
# SHARED: checkpointed row-level work
# =============================================================================
def load_checkpoint(path):
    """Return {key: record} from a JSONL checkpoint, last write wins."""
    done = {}
    if not os.path.exists(path):
        return done
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue          # tolerate a half-written final line
            done[r["key"]] = r
    return done


def needs_run(done, key):
    """A row is outstanding unless it is present AND completed.

    Keying only on presence would treat a row that errored as permanently
    finished, so a single transient CUDA fault would silently become a missing
    judge score.
    """
    return key not in done or done[key].get("status") != "complete"


def capture_environment():
    env = {"python": platform.python_version(), "platform": platform.platform(),
           "numpy": np.__version__, "pandas": pd.__version__}
    for mod in ("scipy", "torch", "transformers", "accelerate", "bitsandbytes",
                "datasets", "bert_score", "sacrebleu", "rouge_score"):
        try:
            env[mod] = __import__(mod).__version__
        except Exception:                                        # noqa: BLE001
            env[mod] = None
    try:
        import torch
        env["cuda"] = torch.version.cuda
        env["gpu"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    except Exception:                                            # noqa: BLE001
        pass
    return env


def write_provenance(stage, extra=None):
    """Append this stage's environment and fingerprints.

    Each stage runs in its own Kaggle session, so a single environment file
    written at the end would describe only the last session's software, not the
    software that actually produced the summaries.
    """
    path = os.path.join(WORK, "provenance.json")
    rec = json.load(open(path)) if os.path.exists(path) else {}
    rec[f"stage_{stage}"] = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "run_label": CFG.run_label,
        "gen_fingerprint": fingerprint(GEN_KEYS),
        "judge_fingerprint": fingerprint(JUDGE_KEYS, globals().get("RUBRIC_HASH", "")),
        "environment": capture_environment(),
        **(extra or {})}
    with open(path, "w") as f:
        json.dump(rec, f, indent=2)
    print(f"recorded provenance for stage {stage}")


def check_generation_provenance():
    """Confirm the summaries on disk came from this generation configuration."""
    path = os.path.join(WORK, "provenance.json")
    want = fingerprint(GEN_KEYS)
    if not os.path.exists(path):
        print("NOTE: no provenance.json from stage 1 (it predates this check).\n"
              "      Falling back to structural validation of the inputs.")
        return
    rec = json.load(open(path))
    got = rec.get("stage_1", {}).get("gen_fingerprint")
    if not got:
        # provenance.json may exist because a later stage wrote it; absence of a
        # stage_1 entry is not verification, so say so rather than imply an audit
        print("NOTE: provenance.json exists but has no stage_1 entry.\n"
              "      Generation provenance is UNVERIFIED; using structural checks.")
        return
    if got != want:
        raise RuntimeError(
            f"generation config fingerprint mismatch: stage 1 produced {got}, "
            f"this session expects {want}. The summaries on disk were made with "
            "different settings; do not mix them.")
    print(f"generation provenance verified ({want})")


def append_checkpoint(path, record):
    with open(path, "a") as f:
        f.write(json.dumps(record) + "\n")
        f.flush()
        os.fsync(f.fileno())


# =============================================================================
# SHARED: statistics
# =============================================================================
from scipy.stats import spearmanr, wilcoxon, rankdata


def sp(x, y):
    """Spearman rho, NaN-dropping; NaN when undefined."""
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = ~(np.isnan(x) | np.isnan(y))
    if m.sum() < 5 or np.unique(x[m]).size < 2 or np.unique(y[m]).size < 2:
        return np.nan
    return spearmanr(x[m], y[m])[0]


def _resid(y, X):
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return y - X @ beta


def partial_spearman(d, metric_col, judge_col, control_cols):
    """Rank both variables, residualise on the controls, correlate the residuals.

    Non-numeric controls are dummy-coded; numeric controls are rank-transformed.
    """
    use = d[[metric_col, judge_col] + list(control_cols)].dropna()
    if len(use) < 10:
        return np.nan
    xr = use[metric_col].rank().to_numpy(float)
    yr = use[judge_col].rank().to_numpy(float)
    parts = [np.ones((len(use), 1))]
    for c in control_cols:
        col = use[c]
        if pd.api.types.is_numeric_dtype(col):
            parts.append(col.rank().to_numpy(float).reshape(-1, 1))
        else:
            dm = pd.get_dummies(col, drop_first=True).astype(float).to_numpy()
            if dm.shape[1]:
                parts.append(dm)
    X = np.column_stack(parts)
    rx, ry = _resid(xr, X), _resid(yr, X)
    if np.std(rx) < 1e-12 or np.std(ry) < 1e-12:
        return np.nan
    return spearmanr(rx, ry)[0]


class ClusterBootstrap:
    """Cluster bootstrap that keeps every row of a drawn cluster together.

    Clusters are drawn WITH replacement and duplicate draws are retained, so a
    cluster selected twice contributes its rows twice. (Filtering with .isin()
    would silently collapse duplicates and produce intervals that are too tight.)

    One fixed set of draws is built once and reused for every statistic, so the
    difference of any two statistics is automatically a paired comparison.
    """

    def __init__(self, df, cluster_col, n_boot=2000, seed=42):
        self.df = df.reset_index(drop=True)
        self.n_boot = n_boot
        clusters = self.df[cluster_col].drop_duplicates().to_numpy()
        pos = {c: self.df.index[self.df[cluster_col] == c].to_numpy() for c in clusters}
        rng = np.random.default_rng(seed)
        k = len(clusters)
        self.n_clusters = k
        self.draws = [np.concatenate([pos[clusters[i]] for i in rng.integers(0, k, k)])
                      for _ in range(n_boot)]

    def run(self, stat_fn, desc=None):
        out = np.empty(self.n_boot)
        for b in range(self.n_boot):
            out[b] = stat_fn(self.df.iloc[self.draws[b]])
        return out


def n_valid(dist):
    return int(np.isfinite(np.asarray(dist, float)).sum())


def ci95(dist, min_fraction=0.95, label=""):
    """Percentile interval, refusing to hide a mostly-undefined distribution.

    nanpercentile will happily return an interval from a handful of replicates.
    An interval built on 200 usable draws out of 2000 is not a 2000-replicate
    interval, so it is rejected rather than quietly reported.
    """
    d = np.asarray(dist, float)
    frac = np.isfinite(d).mean()
    if frac < min_fraction:
        raise ValueError(f"{label or 'statistic'}: only {np.isfinite(d).sum()}/{len(d)} "
                         f"replicates are defined ({frac:.1%}); interval refused")
    lo, hi = np.nanpercentile(d, [2.5, 97.5])
    return float(lo), float(hi)


def boot_p(dist):
    """Two-sided bootstrap p-value with the add-one correction.

    Without it the smallest reportable value is exactly 0, which no finite
    resampling scheme can justify. The floor here is 2/(B+1).
    """
    d = np.asarray(dist, float)
    d = d[np.isfinite(d)]
    if not d.size:
        return np.nan
    lower = ((d <= 0).sum() + 1) / (d.size + 1)
    upper = ((d >= 0).sum() + 1) / (d.size + 1)
    return float(min(1.0, 2 * min(lower, upper)))


def paired_effect(frame, value_col, cond_a, cond_b, case_col="source_row_index"):
    """Matched-pairs separation between two conditions on the same cases.

    Every case is scored under all three conditions, so an independent-groups
    statistic such as ordinary Cliff's delta throws away the pairing and
    overstates the uncertainty. Returns the rank-biserial correlation for
    matched pairs, the proportion of cases where A exceeds B, and Cohen's d_z.
    """
    w = frame.pivot_table(index=case_col, columns="condition", values=value_col,
                          observed=True)
    if cond_a not in w or cond_b not in w:
        return dict(rank_biserial=np.nan, prop_a_higher=np.nan, cohens_dz=np.nan, n_pairs=0)
    w = w[[cond_a, cond_b]].dropna()
    d = (w[cond_a] - w[cond_b]).to_numpy(float)
    nz = d[d != 0]
    if not nz.size:
        return dict(rank_biserial=0.0, prop_a_higher=0.0, cohens_dz=0.0, n_pairs=len(d))
    # Judge scores are discrete, so tied |differences| are common. Sequential
    # ranks would break ties by array order and invent a direction that is not
    # in the data; average ranks are the Wilcoxon convention.
    r = rankdata(np.abs(nz), method="average")
    rb = (r[nz > 0].sum() - r[nz < 0].sum()) / r.sum()
    sd = d.std(ddof=1)
    return dict(rank_biserial=float(rb),
                prop_a_higher=float((d > 0).mean()),
                cohens_dz=float(d.mean() / sd) if sd > 0 else np.nan,
                n_pairs=int(len(d)))


def _self_test_stats():
    rng = np.random.default_rng(0)
    n = 200
    # tier-only association: pooled high, adjusted ~0
    f = [pd.DataFrame({"case": np.arange(n), "condition": t,
                       "x": k + rng.normal(0, .15, n), "y": k + rng.normal(0, .15, n)})
         for k, t in enumerate(CONDITIONS)]
    d1 = pd.concat(f, ignore_index=True)
    assert sp(d1.x, d1.y) > 0.5
    assert abs(partial_spearman(d1, "x", "y", ["condition"])) < 0.10
    # genuine within-condition association survives adjustment
    f = []
    for k, t in enumerate(CONDITIONS):
        z = rng.normal(0, 1, n)
        f.append(pd.DataFrame({"case": np.arange(n), "condition": t,
                               "x": k + z, "y": k + .8 * z + rng.normal(0, .6, n)}))
    d2 = pd.concat(f, ignore_index=True)
    assert sp(d2.x, d2.y) > 0.5 and partial_spearman(d2, "x", "y", ["condition"]) > 0.5
    # shuffled outcome: both near zero
    d3 = d2.copy()
    d3["y"] = rng.permutation(d3.y.to_numpy())
    assert abs(sp(d3.x, d3.y)) < .15 and abs(partial_spearman(d3, "x", "y", ["condition"])) < .15
    # bootstrap keeps clusters intact and retains duplicate draws
    cb = ClusterBootstrap(d2, "case", n_boot=20, seed=1)
    rep = d2.iloc[cb.draws[0]]
    assert len(rep) == len(d2)
    assert rep.groupby("condition").size().eq(n).all()
    assert rep.groupby("case").size().max() > len(CONDITIONS)
    pe = paired_effect(d2.assign(**{"source_row_index": d2.case}), "x",
                       CONDITIONS[2], CONDITIONS[0])
    assert pe["n_pairs"] == n and pe["prop_a_higher"] > .9, "paired effect size"
    # symmetric tied differences must give exactly zero, not an artefact of order
    tied = pd.DataFrame({"source_row_index": [0, 1, 2, 3] * 2,
                         "condition": [CONDITIONS[0]] * 4 + [CONDITIONS[1]] * 4,
                         "v": [2, 2, 1, 1, 1, 1, 2, 2]})
    assert abs(paired_effect(tied, "v", CONDITIONS[0], CONDITIONS[1])["rank_biserial"]) < 1e-12, \
        "tied absolute differences must cancel"
    assert abs(boot_p(np.ones(100))) <= 2 / 101 + 1e-12, "p-value floor is 2/(B+1)"
    print("statistics self-test: passed")


_self_test_stats()


# =============================================================================
# STAGE 1  --  seeded sampling + generation                     GPU, ~7-8 h
# =============================================================================
if stage_runs(1):
    banner("STAGE 1  sampling + generation")
    subprocess.run("pip install -q -U transformers accelerate bitsandbytes",
                   shell=True, check=False)
    import torch
    from transformers import (AutoModelForCausalLM, AutoTokenizer,
                              AutoModelForSeq2SeqLM, BitsAndBytesConfig)
    from tqdm.auto import tqdm

    seed_everything()

    # ---- 1a. seeded random sample, saved so it can be audited and reused ----
    if os.path.exists(F_SELECTED):
        selected = pd.read_csv(F_SELECTED)
        print(f"reusing existing sample: {F_SELECTED}")
    else:
        raw = pd.read_csv(CFG.data_path)
        eligible = (raw.dropna(subset=[CFG.col_input, CFG.col_ref])
                       .reset_index().rename(columns={"index": "source_row_index"}))
        eligible = eligible[eligible[CFG.col_input].astype(str).str.strip().ne("") &
                            eligible[CFG.col_ref].astype(str).str.strip().ne("")]
        need = CFG.demonstration_count + CFG.evaluation_count
        if len(eligible) < need:
            raise ValueError(f"only {len(eligible)} eligible rows, need {need}")
        selected = (eligible.sample(n=need, replace=False, random_state=CFG.seed)
                            .reset_index(drop=True))
        selected["sample_position"] = range(len(selected))
        selected["role"] = "evaluation"
        selected.loc[:CFG.demonstration_count - 1, "role"] = "demonstration"
        selected.to_csv(F_SELECTED, index=False)
        print(f"drew {need} rows from {len(eligible)} eligible with seed {CFG.seed}")

    demos = selected[selected.role == "demonstration"].reset_index(drop=True)
    evals = selected[selected.role == "evaluation"].reset_index(drop=True)

    assert len(demos) == CFG.demonstration_count, f"got {len(demos)} demonstrations"
    assert len(evals) == CFG.evaluation_count, f"got {len(evals)} evaluation cases"
    assert set(demos.source_row_index).isdisjoint(set(evals.source_row_index)), \
        "demonstration cases leaked into the evaluation set"
    assert selected.source_row_index.nunique() == len(selected), "duplicate source rows"
    assert evals[[CFG.col_input, CFG.col_ref]].notna().all().all()
    print(f"sampling checks passed: {len(demos)} demonstrations, {len(evals)} evaluation cases")

    # ---- 1b. load generators ----
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True)
    gtok = AutoTokenizer.from_pretrained(CFG.gen_model)
    gmodel = AutoModelForCausalLM.from_pretrained(
        CFG.gen_model, quantization_config=bnb, device_map="auto")
    gmodel.eval()

    wtok = AutoTokenizer.from_pretrained(CFG.weak_model)
    wmodel = AutoModelForSeq2SeqLM.from_pretrained(
        CFG.weak_model, torch_dtype=torch.float16).to("cuda:0")
    wmodel.eval()
    print(f"loaded {CFG.gen_model} and {CFG.weak_model}")

    SYSTEM = (
        "You are a clinician writing a Brief Hospital Course (BHC). "
        "Given the clinical notes from a hospital stay, write a concise, factual summary "
        "covering: reason for admission, key investigations/findings, treatments given, "
        "the patient's response, and the discharge/follow-up plan. "
        "Write ONLY the BHC summary, in clinical prose. Do not copy the notes verbatim. "
        "Do NOT invent or guess any details. Many records are de-identified - if the age, "
        "sex, dates, or names are missing or redacted, simply omit them rather than making "
        "them up. Only state facts present in the notes."
    )

    def truncation_info(tokenizer, text, limit):
        ids = tokenizer(str(text), add_special_tokens=False, truncation=False)["input_ids"]
        orig = len(ids)
        return {"original_tokens": orig, "used_tokens": min(orig, limit),
                "was_truncated": bool(orig > limit),
                "retained_fraction": (min(orig, limit) / orig) if orig else 1.0}

    def clip(tokenizer, text, limit):
        ids = tokenizer(str(text), truncation=True, max_length=limit)["input_ids"]
        return tokenizer.decode(ids, skip_special_tokens=True)

    def build_zero_shot(notes):
        return [{"role": "system", "content": SYSTEM},
                {"role": "user", "content":
                 f"Clinical notes:\n{clip(gtok, notes, CFG.qwen_input_limit)}"
                 f"\n\nWrite the Brief Hospital Course summary:"}]

    DEMO_MSGS = []
    for _, r in demos.iterrows():
        DEMO_MSGS.append({"role": "user", "content":
                          f"Clinical notes:\n{clip(gtok, r[CFG.col_input], CFG.demonstration_token_limit)}"
                          f"\n\nWrite the Brief Hospital Course summary:"})
        DEMO_MSGS.append({"role": "assistant", "content": str(r[CFG.col_ref]).strip()})

    def build_few_shot(notes):
        return ([{"role": "system", "content": SYSTEM}] + DEMO_MSGS +
                [{"role": "user", "content":
                  f"Clinical notes:\n{clip(gtok, notes, CFG.qwen_input_limit)}"
                  f"\n\nWrite the Brief Hospital Course summary:"}])

    @torch.inference_mode()
    def gen_qwen(messages):
        text = gtok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = gtok(text, return_tensors="pt").to(next(gmodel.parameters()).device)
        pad = gtok.pad_token_id if gtok.pad_token_id is not None else gtok.eos_token_id
        out = gmodel.generate(**inputs, max_new_tokens=CFG.qwen_max_new_tokens,
                              do_sample=False, pad_token_id=pad)
        return gtok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    @torch.inference_mode()
    def gen_weak(notes):
        inp = wtok(str(notes), return_tensors="pt", truncation=True,
                   max_length=CFG.bart_input_limit).to(wmodel.device)
        out = wmodel.generate(**inp, max_new_tokens=CFG.bart_max_new_tokens,
                              num_beams=4, length_penalty=2.0, early_stopping=True)
        return wtok.decode(out[0], skip_special_tokens=True).strip()

    # ---- 1c. generate, checkpointing every case ----
    ckpt = os.path.join(WORK, "gen_checkpoint.jsonl")
    done = load_checkpoint(ckpt)
    print(f"resuming with {len(done)} case(s) already generated")

    t0 = time.time()
    for _, row in tqdm(list(evals.iterrows()), total=len(evals), desc="generating"):
        key = str(row.source_row_index)
        if key in done and done[key].get("status") == "complete":
            continue
        started = time.time()
        try:
            notes, ref = row[CFG.col_input], str(row[CFG.col_ref])
            rec = {"key": key, "status": "complete",
                   "source_row_index": int(row.source_row_index),
                   "sample_position": int(row.sample_position),
                   "reference": ref,
                   "weak_bart": gen_weak(notes),
                   "zero_shot": gen_qwen(build_zero_shot(notes)),
                   "few_shot": gen_qwen(build_few_shot(notes)),
                   "src_trunc_qwen": truncation_info(gtok, notes, CFG.qwen_input_limit),
                   "src_trunc_bart": truncation_info(wtok, notes, CFG.bart_input_limit),
                   "runtime_seconds": round(time.time() - started, 2),
                   "error_message": None}
        except Exception as e:                                   # noqa: BLE001
            rec = {"key": key, "status": "error",
                   "source_row_index": int(row.source_row_index),
                   "error_message": f"{type(e).__name__}: {e}",
                   "runtime_seconds": round(time.time() - started, 2)}
            print(f"\ncase {key} failed: {rec['error_message']}")
        append_checkpoint(ckpt, rec)
        done[key] = rec
    print(f"generation finished in {(time.time()-t0)/60:.1f} min")

    bad = [k for k, v in done.items() if v.get("status") != "complete"]
    if bad:
        raise RuntimeError(f"{len(bad)} case(s) failed; rerun this stage to retry: {bad[:10]}")

    # ---- 1d. assemble the long generation table ----
    rows, trunc_rows = [], []
    for _, row in evals.iterrows():
        r = done[str(row.source_row_index)]
        for cond in CONDITIONS:
            rows.append({"source_row_index": r["source_row_index"],
                         "sample_position": r["sample_position"],
                         "condition": cond, "reference": r["reference"],
                         "generated_summary": r[cond]})
        trunc_rows.append({"source_row_index": r["source_row_index"],
                           **{f"qwen_{k}": v for k, v in r["src_trunc_qwen"].items()},
                           **{f"bart_{k}": v for k, v in r["src_trunc_bart"].items()}})

    gen = pd.DataFrame(rows)
    gen["result_id"] = gen.source_row_index.astype(str) + "__" + gen.condition
    gen["generated_summary"] = gen.generated_summary.astype(str).str.strip()

    assert len(gen) == CFG.evaluation_count * len(CONDITIONS), f"expected 900 rows, got {len(gen)}"
    assert gen.result_id.is_unique, "duplicate case-condition keys"
    assert gen.groupby("condition").size().eq(CFG.evaluation_count).all()
    assert gen.generated_summary.ne("").all(), "an empty summary was generated"
    assert gen.groupby("source_row_index")["reference"].nunique().eq(1).all()

    gen.to_csv(F_GEN, index=False)
    pd.DataFrame(trunc_rows).to_csv(F_TRUNC, index=False)

    tr = pd.DataFrame(trunc_rows)
    print(f"\nsource-note truncation (identical inputs across conditions by design):")
    print(f"  Qwen (limit {CFG.qwen_input_limit:>4} tok): {tr.qwen_was_truncated.mean():.1%} of cases "
          f"truncated, median retained {tr.qwen_retained_fraction.median():.1%}")
    print(f"  BART (limit {CFG.bart_input_limit:>4} tok): {tr.bart_was_truncated.mean():.1%} of cases "
          f"truncated, median retained {tr.bart_retained_fraction.median():.1%}")
    print(f"\nsaved {len(gen)} rows to {F_GEN}")

    write_provenance(1, extra={"n_cases": int(CFG.evaluation_count),
                               "n_rows": int(len(gen))})

    del gmodel, gtok, wmodel, wtok
    gc.collect(); torch.cuda.empty_cache()


# =============================================================================
# STAGE 2  --  automatic metrics                                GPU, ~20 min
# =============================================================================
if stage_runs(2):
    banner("STAGE 2  automatic metrics")
    require(F_GEN, "STAGE 1")
    subprocess.run("pip install -q -U rouge-score sacrebleu bert-score",
                   shell=True, check=False)
    from rouge_score import rouge_scorer
    import sacrebleu
    from bert_score import score as bertscore_fn
    from tqdm.auto import tqdm

    check_generation_provenance()
    gen = pd.read_csv(F_GEN)
    gen["generated_summary"] = gen.generated_summary.astype(str)
    gen["reference"] = gen.reference.astype(str)
    assert len(gen) == CFG.evaluation_count * len(CONDITIONS), \
        f"expected {CFG.evaluation_count*len(CONDITIONS)} rows from stage 1, got {len(gen)}"
    assert gen.source_row_index.nunique() == CFG.evaluation_count
    assert gen.result_id.is_unique
    print(f"scoring {len(gen)} summaries")

    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    r1, r2, rl, bl = [], [], [], []
    for cand, ref in tqdm(list(zip(gen.generated_summary, gen.reference)), desc="rouge/bleu"):
        s = scorer.score(ref, cand)
        r1.append(s["rouge1"].fmeasure)
        r2.append(s["rouge2"].fmeasure)
        rl.append(s["rougeL"].fmeasure)
        bl.append(sacrebleu.sentence_bleu(cand, [ref]).score)     # 0-100
    gen["rouge1"], gen["rouge2"], gen["rougeL"], gen["bleu"] = r1, r2, rl, bl

    # BERTScore in both parameterisations. The rescaled version is the headline
    # figure; the raw version guards against the baseline (fitted on general
    # web text) behaving oddly on clinical prose.
    BS = dict(model_type=CFG.bertscore_model_type, num_layers=CFG.bertscore_num_layers,
              lang=CFG.bertscore_lang)
    _, _, f_resc = bertscore_fn(gen.generated_summary.tolist(), gen.reference.tolist(),
                                rescale_with_baseline=True, verbose=True, **BS)
    _, _, f_raw = bertscore_fn(gen.generated_summary.tolist(), gen.reference.tolist(),
                               rescale_with_baseline=False, verbose=True, **BS)
    gen["bertscore_f1"] = f_resc.numpy()
    gen["bertscore_f1_raw"] = f_raw.numpy()

    for c in ["rouge1", "rouge2", "rougeL"]:
        assert gen[c].between(0, 1).all(), f"{c} outside [0,1]"
    assert gen["bleu"].between(0, 100).all(), "bleu outside [0,100]"
    assert np.isfinite(gen[METRICS].to_numpy()).all(), "non-finite metric value"
    assert gen[METRICS].notna().all().all(), "missing metric value"

    gen.to_csv(F_METRICS, index=False)
    write_provenance(2, extra={"bertscore": BS})
    # stages 2-4 share one session, so hand the GPU back before Qwen-14B loads
    del f_resc, f_raw
    gc.collect()
    try:
        import torch as _t
        _t.cuda.empty_cache()
    except Exception:                                            # noqa: BLE001
        pass
    print("\nmetric means by condition:")
    print(gen.groupby("condition")[METRICS + ["bertscore_f1_raw"]].mean()
             .reindex(CONDITIONS).round(4).to_string())
    print(f"\nsaved {F_METRICS}")


# =============================================================================
# STAGE 3  --  LLM judges on the clinical summaries             GPU, ~2.5 h
# =============================================================================
RUBRIC_CLIN = (
    "You are a strict expert clinician comparing a candidate Brief Hospital Course "
    "(BHC) summary against a reference BHC. You are NOT shown the original clinical "
    "notes, so judge only against the reference. Score THREE dimensions, each 1-5, "
    "using the full scale:\n"
    "CONSISTENCY: are the candidate's statements consistent with the reference, "
    "without contradicting it or adding claims the reference does not support?\n"
    "COMPLETENESS: are the key elements of the hospital course in the reference "
    "also covered by the candidate?\n"
    "COHERENCE: is the candidate well-organized and readable as a clinical summary?\n"
    "Reply in EXACTLY this format, nothing else:\n"
    "CONSISTENCY: <1-5>\nCOMPLETENESS: <1-5>\nCOHERENCE: <1-5>"
)
RUBRIC_GEN = (
    "You are a strict expert evaluator comparing a candidate SUMMARY against a "
    "reference summary. You are NOT shown the source article, so judge only against "
    "the reference. Score THREE dimensions, each 1-5, using the full scale:\n"
    "CONSISTENCY: are the candidate's statements consistent with the reference, "
    "without contradicting it or adding unsupported claims?\n"
    "COMPLETENESS: are the key points of the reference covered?\n"
    "COHERENCE: is it well-organized, fluent and readable?\n"
    "Reply in EXACTLY this format, nothing else:\n"
    "CONSISTENCY: <1-5>\nCOMPLETENESS: <1-5>\nCOHERENCE: <1-5>"
)

JUDGES = [("judge14", CFG.judge_primary), ("judge7", CFG.judge_sensitivity)]

# The rubric text is part of the instrument, so it enters the judging fingerprint.
RUBRIC_HASH = hashlib.sha256((RUBRIC_CLIN + RUBRIC_GEN).encode()).hexdigest()[:12]

# Checkpoint keys embed the judge configuration, so changing the rubric or a
# judge setting starts a fresh set of keys instead of silently reusing scores
# produced by the previous instrument.
JUDGE_RUN_ID = fingerprint(JUDGE_KEYS, RUBRIC_HASH)
SUMMEVAL_RUN_ID = fingerprint(
    JUDGE_KEYS,
    RUBRIC_HASH + f"|{CFG.summeval_dataset}@{CFG.summeval_revision}"
                  f"|{CFG.summeval_articles}x{CFG.summeval_summaries_per_article}"
                  f"|seed{CFG.summeval_seed}")


def _make_judge(model, tok):
    import torch

    def clip(text, limit):
        ids = tok(str(text), truncation=True, max_length=limit)["input_ids"]
        return tok.decode(ids, skip_special_tokens=True)

    @torch.inference_mode()
    def run(rubric, ref_label, ref, cand_label, cand):
        user = (rubric + f"\n\n{ref_label}:\n" + clip(ref, CFG.judge_field_limit) +
                f"\n\n{cand_label}:\n" + clip(cand, CFG.judge_field_limit) + "\n\nScores:")
        text = tok.apply_chat_template([{"role": "user", "content": user}],
                                       tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors="pt").to(next(model.parameters()).device)
        pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
        out = model.generate(**inputs, max_new_tokens=CFG.judge_max_new_tokens,
                             do_sample=False, pad_token_id=pad)
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    return run


def _model_fingerprint(model_id, tok, model):
    info = {"model_identifier": model_id, "tokenizer_identifier": model_id,
            "quantization_type": "nf4-4bit", "compute_dtype": "float16",
            "device_map": "auto", "model_revision": None, "tokenizer_revision": None}
    try:
        from huggingface_hub import HfApi
        info["model_revision"] = HfApi().model_info(model_id).sha
        info["tokenizer_revision"] = info["model_revision"]
    except Exception:                                            # noqa: BLE001
        pass
    return info


if stage_runs(3):
    banner("STAGE 3  judging the clinical summaries")
    require(F_METRICS, "STAGE 2")
    subprocess.run("pip install -q -U transformers accelerate bitsandbytes",
                   shell=True, check=False)
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from tqdm.auto import tqdm

    seed_everything()
    check_generation_provenance()
    gen = pd.read_csv(F_METRICS)
    gen["generated_summary"] = gen.generated_summary.astype(str)
    gen["reference"] = gen.reference.astype(str)

    done = load_checkpoint(F_JUDGE_RAW)
    print(f"resuming with {len(done)} judgement(s) already recorded")

    fingerprints = {}
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True)

    for tag, model_id in JUDGES:
        todo = [r for _, r in gen.iterrows()
                if needs_run(done, f"{JUDGE_RUN_ID}|{tag}|{r.result_id}")]
        if not todo:
            print(f"{tag}: already complete, skipping")
            continue
        print(f"\nloading {tag}: {model_id}  ({len(todo)} to judge)")
        tok = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb, device_map="auto")
        model.eval()
        fingerprints[tag] = _model_fingerprint(model_id, tok, model)
        judge = _make_judge(model, tok)

        for r in tqdm(todo, desc=tag):
            key = f"{JUDGE_RUN_ID}|{tag}|{r.result_id}"
            started = time.time()
            try:
                raw = judge(RUBRIC_CLIN, "REFERENCE BHC", r.reference,
                            "SUMMARY TO RATE", r.generated_summary)
                status = "complete"
                err = None
            except Exception as e:                               # noqa: BLE001
                raw, status, err = "", "error", f"{type(e).__name__}: {e}"
            append_checkpoint(F_JUDGE_RAW, {
                "key": key, "judge": tag, "result_id": r.result_id,
                "source_row_index": int(r.source_row_index), "condition": r.condition,
                "raw_output": raw, "status": status, "error_message": err,
                "runtime_seconds": round(time.time() - started, 2)})
        del model, tok
        gc.collect(); torch.cuda.empty_cache()
        print(f"{tag} done and unloaded")

    if fingerprints:
        with open(os.path.join(WORK, "model_revisions.json"), "w") as f:
            json.dump(fingerprints, f, indent=2)

    # ---- parse every raw output and merge on keys, never on row order ----
    raws = pd.DataFrame(load_checkpoint(F_JUDGE_RAW).values())
    raws = raws[raws.key.str.startswith(JUDGE_RUN_ID)].reset_index(drop=True)
    parsed = raws.raw_output.fillna("").apply(parse_judge_output)
    raws[["s1", "s2", "s3", "mean", "parse_status"]] = pd.DataFrame(
        parsed.tolist(), index=raws.index)

    print("\nparse coverage:")
    for tag, _ in JUDGES:
        sub = raws[raws.judge == tag]
        ok_rate = sub.parse_status.str.startswith("ok").mean()
        print(f"  {tag:8s} {len(sub):4d} outputs, {ok_rate:.1%} parsed "
              f"({sub.parse_status.value_counts().to_dict()})")

    master = pd.read_csv(F_METRICS)
    for tag, _ in JUDGES:
        sub = raws[raws.judge == tag][["result_id", "s1", "s2", "s3", "mean", "raw_output", "parse_status"]]
        sub = sub.drop_duplicates("result_id", keep="last").rename(columns={
            "s1": f"{tag}_consistency", "s2": f"{tag}_completeness", "s3": f"{tag}_coherence",
            "mean": f"{tag}_mean", "raw_output": f"{tag}_raw", "parse_status": f"{tag}_parse_status"})
        master = master.merge(sub, on="result_id", how="left", validate="one_to_one")
        # the composite must be exactly the mean of its three components
        comp = master[[f"{tag}_{d}" for d in DIMS3]].mean(axis=1)
        ok = master[f"{tag}_mean"].notna()
        assert np.allclose(comp[ok], master.loc[ok, f"{tag}_mean"]), \
            f"{tag} composite does not equal the mean of its three dimensions"
        assert master.loc[ok, f"{tag}_mean"].between(1, 5).all(), f"{tag}_mean outside [1,5]"

    # Stop here rather than let stage 5 analyse a partly missing outcome variable.
    for tag, _ in JUDGES:
        cov = master[f"{tag}_mean"].notna().mean()
        okr = master[f"{tag}_parse_status"].fillna("").str.startswith("ok").mean()
        floor = 0.99 if tag == "judge14" else 0.95
        if cov < floor or okr < floor:
            raise RuntimeError(
                f"{tag}: only {cov:.1%} of rows have a score and {okr:.1%} parsed "
                f"(floor {floor:.0%}). Rerun this stage - failed rows are retried "
                f"automatically. If it persists, inspect {F_JUDGE_RAW}.")
        print(f"  {tag} coverage {cov:.1%}, parsed {okr:.1%}")

    master["cand_words"] = master.generated_summary.astype(str).str.split().apply(len)
    master["ref_words"] = master.reference.astype(str).str.split().apply(len)
    master["len_ratio"] = master.cand_words / master.ref_words.replace(0, np.nan)

    assert len(master) == CFG.evaluation_count * len(CONDITIONS)
    assert master.result_id.is_unique
    master.to_csv(F_MASTER, index=False)
    print(f"\nsaved master file: {F_MASTER}  ({master.shape[0]} rows x {master.shape[1]} cols)")

    print("\njudge means by condition:")
    print(master.groupby("condition")[[f"{t}_mean" for t, _ in JUDGES]]
                .agg(["mean", "std"]).reindex(CONDITIONS).round(3).to_string())

    # ---- stratified audit sheet, both judges, filled in by hand ----
    sheets = []
    for tag, _ in JUDGES:
        a = (master[master[f"{tag}_parse_status"].notna()]
             .groupby("condition", group_keys=False)
             .sample(n=15, random_state=CFG.seed))
        a = a[["result_id", "condition", "generated_summary"]].copy()
        a["judge"] = tag
        a["raw_output"] = master.loc[a.index, f"{tag}_raw"].values
        for d in DIMS3:
            a[d] = master.loc[a.index, f"{tag}_{d}"].values
        a["mean"] = master.loc[a.index, f"{tag}_mean"].values
        a["parse_status"] = master.loc[a.index, f"{tag}_parse_status"].values
        sheets.append(a)
    audit = pd.concat(sheets, ignore_index=True)
    for c in ["parser_correct", "three_scores_present", "score_order_correct",
              "unexpected_digits", "manual_comment"]:
        audit[c] = ""
    audit.to_csv(os.path.join(WORK, "judge_audit_sheet.csv"), index=False)
    write_provenance(3, extra={"rubric_sha": RUBRIC_HASH, "judges": [t for t, _ in JUDGES]})
    print(f"wrote judge_audit_sheet.csv ({len(audit)} rows, both judges) for manual review")


# =============================================================================
# STAGE 4  --  SummEval judge validation                  GPU + internet, ~2.5 h
# =============================================================================
if stage_runs(4):
    banner("STAGE 4  SummEval validation against human expert ratings")
    subprocess.run("pip install -q -U transformers accelerate bitsandbytes datasets "
                   "rouge-score sacrebleu bert-score", shell=True, check=False)
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from datasets import load_dataset
    from rouge_score import rouge_scorer
    import sacrebleu
    from bert_score import score as bertscore_fn
    from tqdm.auto import tqdm

    seed_everything()

    # ---- 4a. sample WHOLE articles, keeping all system summaries per article --
    # G-Eval reports summary-level correlations computed within each source
    # article. That statistic only exists if every sampled article keeps its
    # full set of system summaries, so we sample articles, not summaries.
    ds = load_dataset(CFG.summeval_dataset, revision=CFG.summeval_revision, split="test")
    try:
        from huggingface_hub import HfApi
        resolved = HfApi().dataset_info(CFG.summeval_dataset,
                                        revision=CFG.summeval_revision).sha
        print(f"SummEval resolved revision: {resolved}")
    except Exception:                                            # noqa: BLE001
        resolved = None
    DIMS = [d for d in ["coherence", "consistency", "fluency", "relevance"]
            if d in ds.column_names]
    print(f"human dimensions: {DIMS}")

    rng = np.random.default_rng(CFG.summeval_seed)
    n_art_total = len(ds)
    k = min(CFG.summeval_articles, n_art_total)
    art_ids = np.sort(rng.choice(n_art_total, size=k, replace=False))
    print(f"sampled {k} of {n_art_total} articles (seed {CFG.summeval_seed})")

    rows = []
    for a in art_ids:
        ex = ds[int(a)]
        refs = [r for r in (ex.get("human_summaries") or []) if isinstance(r, str) and r.strip()]
        for si, summ in enumerate(ex["machine_summaries"]):
            per = {d: ex[d][si] for d in DIMS}
            rows.append({"article": int(a), "system_index": si,
                         "summary": summ, "reference": refs[0] if refs else "",
                         "refs_all": json.dumps(refs),
                         "human_overall": float(np.mean(list(per.values()))),
                         **{f"human_{d}": per[d] for d in DIMS}})
    val = pd.DataFrame(rows)
    val["se_id"] = val.article.astype(str) + "__" + val.system_index.astype(str)
    per = CFG.summeval_summaries_per_article
    assert val.se_id.is_unique
    assert val.article.nunique() == k, f"expected {k} articles, got {val.article.nunique()}"
    assert val.groupby("article").size().eq(per).all(), \
        f"every article must contribute {per} system summaries"
    assert len(val) == k * per, f"expected {k*per} summaries, got {len(val)}"
    print(f"{len(val)} summaries = {k} articles x {per} systems "
          f"(whole articles are required for the summary-level statistic)")

    # ---- 4b. judge them, checkpointed ----
    done = load_checkpoint(F_SUMM_RAW)
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True)
    for tag, model_id in JUDGES:
        todo = [r for _, r in val.iterrows()
                if needs_run(done, f"{SUMMEVAL_RUN_ID}|{tag}|{r.se_id}")]
        if not todo:
            print(f"{tag}: already complete, skipping")
            continue
        print(f"\nloading {tag}: {model_id}  ({len(todo)} to judge)")
        tok = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb, device_map="auto")
        model.eval()
        judge = _make_judge(model, tok)
        for r in tqdm(todo, desc=f"summeval:{tag}"):
            started = time.time()
            try:
                raw = judge(RUBRIC_GEN, "REFERENCE SUMMARY", r.reference,
                            "SUMMARY TO RATE", r.summary)
                status, err = "complete", None
            except Exception as e:                               # noqa: BLE001
                raw, status, err = "", "error", f"{type(e).__name__}: {e}"
            append_checkpoint(F_SUMM_RAW, {
                "key": f"{SUMMEVAL_RUN_ID}|{tag}|{r.se_id}", "judge": tag, "se_id": r.se_id,
                "raw_output": raw, "status": status, "error_message": err,
                "runtime_seconds": round(time.time() - started, 2)})
        del model, tok
        gc.collect(); torch.cuda.empty_cache()
        print(f"{tag} done and unloaded")

    raws = pd.DataFrame(load_checkpoint(F_SUMM_RAW).values())
    raws = raws[raws.key.str.startswith(SUMMEVAL_RUN_ID)].reset_index(drop=True)
    parsed = raws.raw_output.fillna("").apply(parse_judge_output)
    raws[["s1", "s2", "s3", "mean", "parse_status"]] = pd.DataFrame(
        parsed.tolist(), index=raws.index)
    for tag, _ in JUDGES:
        sub = raws[raws.judge == tag]
        if len(sub):
            print(f"  {tag}: {sub.parse_status.str.startswith('ok').mean():.1%} parsed")
        s = sub[["se_id", "mean", "raw_output", "parse_status"]].drop_duplicates("se_id", keep="last")
        s = s.rename(columns={"mean": f"{tag}", "raw_output": f"{tag}_raw",
                              "parse_status": f"{tag}_parse_status"})
        val = val.merge(s, on="se_id", how="left", validate="one_to_one")

    for tag, _ in JUDGES:
        cov = val[tag].notna().mean() if tag in val.columns else 0.0
        floor = 0.99 if tag == "judge14" else 0.95
        if cov < floor:
            raise RuntimeError(
                f"SummEval {tag} coverage {cov:.1%} is below the {floor:.0%} floor. "
                f"Rerun stage 4 - failed rows are retried automatically. "
                f"If it persists, inspect {F_SUMM_RAW}.")
        print(f"  {tag} coverage {cov:.1%}")

    # ---- 4c. the same metrics, multi-reference and single-reference ----
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    recs = []
    for cand, refs_json, ref1 in tqdm(list(zip(val.summary.astype(str), val.refs_all,
                                               val.reference.astype(str))), desc="metrics"):
        refs = json.loads(refs_json) or [ref1]
        best = {k2: max(scorer.score(r, cand)[k2].fmeasure for r in refs)
                for k2 in ("rouge1", "rouge2", "rougeL")}
        recs.append({**best,
                     "bleu": sacrebleu.sentence_bleu(cand, refs).score,
                     "rouge1_single": scorer.score(ref1, cand)["rouge1"].fmeasure,
                     "bleu_single": sacrebleu.sentence_bleu(cand, [ref1]).score})
    val = pd.concat([val, pd.DataFrame(recs, index=val.index)], axis=1)

    BS = dict(model_type=CFG.bertscore_model_type, num_layers=CFG.bertscore_num_layers,
              lang=CFG.bertscore_lang)
    cands = val.summary.astype(str).tolist()
    refs_ll = [json.loads(r) or [t] for r, t in zip(val.refs_all, val.reference.astype(str))]
    _, _, fm = bertscore_fn(cands, refs_ll, rescale_with_baseline=True, verbose=True, **BS)
    _, _, fs = bertscore_fn(cands, val.reference.astype(str).tolist(),
                            rescale_with_baseline=True, verbose=True, **BS)
    val["bertscore_f1"] = fm.numpy()
    val["bertscore_f1_single"] = fs.numpy()

    val.to_csv(F_SUMMEVAL, index=False)
    write_provenance(4, extra={"summeval_dataset": CFG.summeval_dataset,
                               "summeval_resolved_revision": resolved,
                               "n_articles": int(k), "n_summaries": int(len(val)),
                               "bertscore": BS})
    print(f"\nsaved {F_SUMMEVAL} ({len(val)} rows)")
    for tag, _ in JUDGES:
        if tag in val.columns:
            print(f"  {tag} vs human_overall (dataset-level): rho = {sp(val[tag], val.human_overall):+.3f}")


# =============================================================================
# STAGE 5  --  analysis: every table and figure generated from the master file
# =============================================================================
PRIMARY, SENS = "judge14_mean", "judge7_mean"


def article_rho_series(frame, mcol, hcol, group="article", min_n=3, zero_variance="drop"):
    """Per-article Spearman rho INDEXED BY ARTICLE, NaN where undefined.

    Keeping the article index (rather than returning a compacted array) is what
    lets every human dimension be resampled with the same article draw. Without
    it each dimension gets its own draw, the article-level pairing is lost, and
    the resulting interval is far too narrow.
    """
    vals = {}
    for aid, g in frame.groupby(group):
        if len(g) < min_n or g[hcol].nunique() < 2:
            vals[aid] = np.nan
        elif g[mcol].nunique() < 2:
            vals[aid] = 0.0 if zero_variance == "zero" else np.nan
        else:
            r = spearmanr(g[mcol], g[hcol])[0]
            vals[aid] = r if not np.isnan(r) else (0.0 if zero_variance == "zero" else np.nan)
    return pd.Series(vals, dtype=float)


def article_rhos(frame, mcol, hcol, group="article", min_n=3, zero_variance="drop"):
    """Per-article rho with undefined articles dropped."""
    return article_rho_series(frame, mcol, hcol, group, min_n, zero_variance).dropna().to_numpy()


def summary_level_rho(frame, mcol, hcol, group="article", min_n=3, zero_variance="drop"):
    """G-Eval's summary-level statistic: correlate within each source article,
    then average across articles.

    zero_variance='drop' matches the usual reporting convention; 'zero' scores a
    tied instrument as rho=0 instead of excluding the article, which is the
    conservative treatment for a coarse judge that often gives one value to
    every summary of an article.
    """
    vals = article_rhos(frame, mcol, hcol, group, min_n, zero_variance)
    return (float(vals.mean()) if vals.size else np.nan), int(vals.size)


if stage_runs(5):
    banner("STAGE 5  analysis")
    require(F_MASTER, "STAGE 3")
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    M = pd.read_csv(F_MASTER)
    M["condition"] = pd.Categorical(M.condition, categories=CONDITIONS, ordered=True)
    NB = CFG.bootstrap_replicates
    CB = ClusterBootstrap(M, "source_row_index", n_boot=NB, seed=CFG.seed)
    print(f"{len(M)} rows, {CB.n_clusters} cases, {NB} shared case-clustered draws")

    tables, dists = {}, {}

    # ---- Table 1: descriptives -------------------------------------------
    t1 = (M.groupby("condition", observed=True)
            .agg(n=("result_id", "size"),
                 words_mean=("cand_words", "mean"), words_sd=("cand_words", "std"),
                 len_ratio=("len_ratio", "median"),
                 **{m: (m, "mean") for m in METRICS},
                 judge_mean=(PRIMARY, "mean"), judge_sd=(PRIMARY, "std"))
            .reindex(CONDITIONS).round(4))
    tables["table1_descriptives"] = t1.reset_index()

    # ---- Table 2: how well does each instrument separate the systems? ------
    # Context for everything that follows: an instrument cannot be confounded by
    # a between-system gap it never detected in the first place.
    rows = []
    for name in METRICS + [PRIMARY]:
        rec = dict(instrument=name)
        for t in CONDITIONS:
            rec[f"{t}_mean"] = M.loc[M.condition == t, name].mean()
        for good in ["zero_shot", "few_shot"]:
            pe = paired_effect(M, name, good, "weak_bart")
            rec[f"rb_{good}_vs_weak"] = pe["rank_biserial"]
            rec[f"prop_{good}_higher"] = pe["prop_a_higher"]
            rec[f"dz_{good}_vs_weak"] = pe["cohens_dz"]
        rows.append(rec)
    tables["table2_system_separation"] = pd.DataFrame(rows).round(4)

    # ---- Table 3: pooled vs adjusted --------------------------------------
    rows = []
    for m in METRICS:
        pooled = sp(M[m], M[PRIMARY])
        adj = partial_spearman(M, m, PRIMARY, ["condition"])
        adjl = partial_spearman(M, m, PRIMARY, ["condition", "len_ratio"])
        dp = CB.run(lambda d, m=m: sp(d[m], d[PRIMARY]))
        da = CB.run(lambda d, m=m: partial_spearman(d, m, PRIMARY, ["condition"]))
        dl = CB.run(lambda d, m=m: partial_spearman(d, m, PRIMARY, ["condition", "len_ratio"]))
        dists[f"pooled_{m}"], dists[f"adjusted_{m}"], dists[f"adjlen_{m}"] = dp, da, dl
        pl, ph = ci95(dp, label=f"pooled {m}")
        al, ah = ci95(da, label=f"adjusted {m}")
        ll, lh = ci95(dl, label=f"adjusted+length {m}")
        rows.append(dict(metric=m, pooled=pooled, pooled_lo=pl, pooled_hi=ph,
                         adjusted=adj, adjusted_lo=al, adjusted_hi=ah,
                         adj_plus_length=adjl, adjlen_lo=ll, adjlen_hi=lh,
                         pct_of_pooled_lost=100 * (1 - adj / pooled) if pooled else np.nan,
                         n_valid_pooled=n_valid(dp), n_valid_adjusted=n_valid(da),
                         n_boot=NB))
    tables["table3_pooled_vs_adjusted"] = pd.DataFrame(rows).round(4)

    # ---- Table 4: within-condition ----------------------------------------
    rows = []
    for t in CONDITIONS:
        for m in METRICS:
            f = lambda d, t=t, m=m: sp(d.loc[d.condition == t, m], d.loc[d.condition == t, PRIMARY])
            dist = CB.run(f)
            lo, hi = ci95(dist, label=f"within {t} {m}")
            rows.append(dict(condition=t, metric=m, rho=f(M), lo=lo, hi=hi,
                             ci_excludes_zero=bool(lo > 0 or hi < 0),
                             n_valid=n_valid(dist), n_boot=NB))
    tables["table4_within_condition"] = pd.DataFrame(rows).round(4)

    # ---- Table 5: paired contrasts between metrics -------------------------
    rows = []
    for i, a in enumerate(METRICS):
        for b in METRICS[i + 1:]:
            dist = dists[f"adjusted_{a}"] - dists[f"adjusted_{b}"]   # shared draws => paired
            lo, hi = ci95(dist, label=f"{a} vs {b}")
            rows.append(dict(contrast=f"{a} - {b}",
                             diff=partial_spearman(M, a, PRIMARY, ["condition"])
                                  - partial_spearman(M, b, PRIMARY, ["condition"]),
                             lo=lo, hi=hi, p=boot_p(dist),
                             ci_excludes_zero=bool(lo > 0 or hi < 0)))
    tables["table5_metric_contrasts"] = pd.DataFrame(rows).round(4)

    # ---- Table 6: weak tier vs good tiers, paired --------------------------
    rows = []
    for m in METRICS:
        for good in ["zero_shot", "few_shot"]:
            f = lambda d, m=m, g=good: (
                sp(d.loc[d.condition == "weak_bart", m], d.loc[d.condition == "weak_bart", PRIMARY])
                - sp(d.loc[d.condition == g, m], d.loc[d.condition == g, PRIMARY]))
            dist = CB.run(f)
            lo, hi = ci95(dist, label=f"{m} weak vs {good}")
            rows.append(dict(metric=m, good_tier=good, diff=f(M), lo=lo, hi=hi,
                             p=boot_p(dist), ci_excludes_zero=bool(lo > 0 or hi < 0)))
    tables["table6_weak_vs_good"] = pd.DataFrame(rows).round(4)

    # ---- Table 7: divergence / risk difference -----------------------------
    rows = []
    for t in CONDITIONS:
        for m in METRICS:
            def rd(d, t=t, m=m):
                s = d[d.condition == t][[m, PRIMARY]].dropna()
                if len(s) < 10:
                    return np.nan
                high = s[m] >= s[m].quantile(CFG.high_quantile)
                if not high.sum():
                    return np.nan
                low = s[PRIMARY] < CFG.divergence_cutoff
                return low[high].mean() - low.mean()
            dist = CB.run(rd)
            lo, hi = ci95(dist, label=f"divergence {t} {m}")
            s = M[M.condition == t][[m, PRIMARY]].dropna()
            high = s[m] >= s[m].quantile(CFG.high_quantile)
            low = s[PRIMARY] < CFG.divergence_cutoff
            rows.append(dict(condition=t, metric=m, n_high=int(high.sum()),
                             base_rate=low.mean(), divergence=low[high].mean(),
                             risk_difference=rd(M), lo=lo, hi=hi,
                             enriched=bool(lo > 0)))
    tables["table7_divergence"] = pd.DataFrame(rows).round(4)

    # ---- Table 8: judge behaviour, granularity and length ------------------
    rows = []
    vc = M[PRIMARY].value_counts(normalize=True)
    for t in CONDITIONS:
        s = M[M.condition == t]
        rows.append(dict(condition=t,
                         judge_vs_words=sp(s.cand_words, s[PRIMARY]),
                         judge_vs_len_ratio=sp(s.len_ratio, s[PRIMARY]),
                         judge_distinct_values=int(s[PRIMARY].nunique())))
    t8 = pd.DataFrame(rows).round(4)
    t8.attrs["pooled_distinct_values"] = int(M[PRIMARY].nunique())
    t8.attrs["modal_share"] = float(vc.max())
    tables["table8_judge_behaviour"] = t8

    # ---- Table 9: per-dimension judge correlations -------------------------
    rows = []
    for dim in list(DIMS3) + ["mean"]:
        col = f"judge14_{dim}"
        if col not in M.columns:
            continue
        for m in METRICS:
            rows.append(dict(dimension=dim, metric=m,
                             pooled=sp(M[m], M[col]),
                             adjusted=partial_spearman(M, m, col, ["condition"])))
    tables["table9_judge_dimensions"] = pd.DataFrame(rows).round(4)

    # ---- Table 10: sensitivity judge ---------------------------------------
    rows = []
    for m in METRICS:
        ds_ = CB.run(lambda d, m=m: partial_spearman(d, m, SENS, ["condition"]))
        dd_ = dists[f"adjusted_{m}"] - ds_          # shared draws, so this is paired
        slo, shi = ci95(ds_, label=f"sens {m}")
        dlo, dhi = ci95(dd_, label=f"judge diff {m}")
        rows.append(dict(metric=m,
                         adjusted_primary=partial_spearman(M, m, PRIMARY, ["condition"]),
                         adjusted_sens=partial_spearman(M, m, SENS, ["condition"]),
                         sens_lo=slo, sens_hi=shi,
                         diff_primary_minus_sens=partial_spearman(M, m, PRIMARY, ["condition"])
                                                 - partial_spearman(M, m, SENS, ["condition"]),
                         diff_lo=dlo, diff_hi=dhi, diff_p=boot_p(dd_),
                         judges_differ=bool(dlo > 0 or dhi < 0)))
    t10 = pd.DataFrame(rows).round(4)
    dj = CB.run(lambda d: sp(d[PRIMARY], d[SENS]))
    jlo, jhi = ci95(dj, label="inter-judge")
    t10.attrs["inter_judge_rho"] = float(sp(M[PRIMARY], M[SENS]))
    t10.attrs["inter_judge_ci"] = (round(jlo, 4), round(jhi, 4))
    tables["table10_judge_sensitivity"] = t10

    # ---- Table 11: is few-shot really better than zero-shot? ---------------
    pair = M.pivot_table(index="source_row_index", columns="condition",
                         values=PRIMARY, observed=True).dropna()
    dif = pair["few_shot"] - pair["zero_shot"]
    rng = np.random.default_rng(CFG.seed)
    md = np.array([dif.to_numpy()[rng.integers(0, len(dif), len(dif))].mean() for _ in range(NB)])
    lo, hi = ci95(md, label="few vs zero")
    try:
        wp = float(wilcoxon(pair["few_shot"], pair["zero_shot"]).pvalue)
    except ValueError:
        wp = float("nan")
    tables["table11_fewshot_vs_zeroshot"] = pd.DataFrame([dict(
        zero_shot_mean=pair["zero_shot"].mean(), few_shot_mean=pair["few_shot"].mean(),
        mean_difference=dif.mean(), lo=lo, hi=hi,
        cohens_d_paired=dif.mean() / dif.std(ddof=1), wilcoxon_p=wp)]).round(4)

    # ---- SummEval tables, if stage 4 has run -------------------------------
    if os.path.exists(F_SUMMEVAL):
        V = pd.read_csv(F_SUMMEVAL)
        dims = [c for c in V.columns if c.startswith("human_") and c != "human_overall"]
        instruments = [(m, m) for m in METRICS] + [("judge14", "judge14"), ("judge7", "judge7")]
        instruments = [(n, c) for n, c in instruments if c in V.columns]

        VB = ClusterBootstrap(V, "article", n_boot=NB, seed=CFG.seed)
        rows = []
        for name, c in instruments:
            dist = VB.run(lambda d, c=c: sp(d[c], d.human_overall))
            lo, hi = ci95(dist, label=f"summeval {name}")
            dists[f"summeval_{name}"] = dist
            rows.append(dict(instrument=name, rho_dataset_level=sp(V[c], V.human_overall),
                             lo=lo, hi=hi))
        tables["table12_summeval_dataset_level"] = pd.DataFrame(rows).round(4)

        # G-Eval-comparable statistic, both zero-variance conventions
        rows = []
        for name, c in instruments:
            rec = {"instrument": name}
            for mode in ("drop", "zero"):
                per = [summary_level_rho(V, c, d, zero_variance=mode)[0] for d in dims]
                rec[f"avg_{mode}"] = float(np.mean(per))
                for d, r in zip(dims, per):
                    rec[f"{d.replace('human_','')}_{mode}"] = r
            # articles x dimensions, NaN where an article is unusable for that
            # dimension. One draw of article rows per replicate keeps the four
            # dimensions paired, exactly as they are paired in the data.
            rho_matrix = pd.concat(
                {d: article_rho_series(V, c, d, zero_variance="drop") for d in dims},
                axis=1).to_numpy(float)
            brng = np.random.default_rng(CFG.seed)
            n_art = rho_matrix.shape[0]
            dist = np.empty(NB)
            with np.errstate(invalid="ignore"):
                for b in range(NB):
                    sampled = rho_matrix[brng.integers(0, n_art, n_art)]
                    per_dim = np.nanmean(sampled, axis=0)
                    dist[b] = np.nanmean(per_dim) if np.isfinite(per_dim).any() else np.nan
            lo, hi = ci95(dist, label=f"summary-level {name}")
            rec["avg_drop_lo"], rec["avg_drop_hi"] = lo, hi
            for d in dims:
                rec[f"n_articles_{d.replace('human_','')}"] = summary_level_rho(V, c, d)[1]
            rec["n_valid"] = n_valid(dist)
            rows.append(rec)
        t13 = pd.DataFrame(rows).round(4)
        t13.attrs["note"] = (
            "G-Eval reports 0.514 on this statistic with GPT-4. That figure uses a "
            "different judge, prompt, subset and dataset revision, so treat it as "
            "literature context rather than a benchmark this run is scored against.")
        tables["table13_summeval_summary_level"] = t13

        rows = []
        for name, c in instruments:
            if name.startswith("judge"):
                continue
            dist = dists["summeval_judge14"] - dists[f"summeval_{name}"]
            lo, hi = ci95(dist, label=f"judge vs {name}")
            rows.append(dict(metric=name,
                             rho_metric=sp(V[c], V.human_overall),
                             rho_judge=sp(V.judge14, V.human_overall),
                             diff=sp(V.judge14, V.human_overall) - sp(V[c], V.human_overall),
                             lo=lo, hi=hi, p=boot_p(dist),
                             ci_excludes_zero=bool(lo > 0 or hi < 0)))
        tables["table14_judge_vs_metrics_on_humans"] = pd.DataFrame(rows).round(4)

    # ---- figures ----------------------------------------------------------
    t3 = tables["table3_pooled_vs_adjusted"]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    y = np.arange(len(t3))
    ax.errorbar(t3.pooled, y + .16, xerr=[t3.pooled - t3.pooled_lo, t3.pooled_hi - t3.pooled],
                fmt="o", capsize=3, label="pooled across systems")
    ax.errorbar(t3.adjusted, y - .16, xerr=[t3.adjusted - t3.adjusted_lo, t3.adjusted_hi - t3.adjusted],
                fmt="s", capsize=3, label="adjusted for system")
    ax.set_yticks(y); ax.set_yticklabels(t3.metric)
    ax.axvline(0, color="grey", lw=.8)
    ax.set_xlabel("Spearman rho with the LLM judge (case-clustered 95% CI)")
    ax.legend(frameon=False); ax.set_title("Agreement before and after adjusting for system")
    fig.tight_layout(); fig.savefig(f"{OUTPUTS}/figure_pooled_vs_adjusted.png", dpi=200)
    plt.close(fig)

    t4 = tables["table4_within_condition"]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for off, t in zip([-.22, 0, .22], CONDITIONS):
        s = t4[t4.condition == t]
        ax.errorbar(s.rho, np.arange(len(METRICS)) + off,
                    xerr=[s.rho - s.lo, s.hi - s.rho], fmt="o", capsize=3, label=t)
    ax.set_yticks(range(len(METRICS))); ax.set_yticklabels(METRICS)
    ax.axvline(0, color="grey", lw=.8)
    ax.set_xlabel("Spearman rho with the LLM judge (case-clustered 95% CI)")
    ax.legend(frameon=False); ax.set_title("Agreement within each generation system")
    fig.tight_layout(); fig.savefig(f"{OUTPUTS}/figure_within_condition.png", dpi=200)
    plt.close(fig)

    fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)
    for ax, t in zip(axes, CONDITIONS):
        s = M[M.condition == t]
        ax.scatter(s.bertscore_f1, s[PRIMARY], s=12, alpha=.5)
        ax.set_title(t); ax.set_xlabel("BERTScore F1")
    axes[0].set_ylabel("LLM judge (1-5)")
    fig.suptitle("BERTScore against judged quality, by system")
    fig.tight_layout(); fig.savefig(f"{OUTPUTS}/figure_bertscore_scatter.png", dpi=200)
    plt.close(fig)

    # ---- write everything --------------------------------------------------
    for name, tab in tables.items():
        tab.to_csv(f"{OUTPUTS}/{name}.csv", index=False)
        print(f"\n### {name}")
        print(tab.to_string(index=False))
        for k2, v2 in getattr(tab, "attrs", {}).items():
            print(f"  [{k2}] {v2}")

    pd.DataFrame(dists).to_csv(f"{OUTPUTS}/bootstrap_distributions.csv", index=False)

    # Public export: one CSV per table plus an index. Concatenating fourteen
    # tables with different columns produces a mostly-empty sheet nobody can read.
    pub = os.path.join(WORK, "public_results")
    os.makedirs(pub, exist_ok=True)
    index = []
    for name, tab in tables.items():
        tab.to_csv(f"{pub}/{name}.csv", index=False)
        index.append({"table": name, "file": f"{name}.csv",
                      "rows": int(len(tab)), "columns": list(tab.columns),
                      "notes": {k: str(v) for k, v in getattr(tab, "attrs", {}).items()}})
    with open(f"{pub}/index.json", "w") as f:
        json.dump({"run_label": CFG.run_label, "n_boot": NB,
                   "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
                   "gen_fingerprint": fingerprint(GEN_KEYS), "tables": index}, f, indent=2)
    write_provenance(5, extra={"n_tables": len(tables), "n_boot": NB})
    print(f"\nwrote {len(tables)} tables, 3 figures and bootstrap distributions to {OUTPUTS}")
    print(f"public aggregate export (no clinical text): {pub}/")


# =============================================================================
# STAGE 6  --  independent verification, environment freeze, manifest
# =============================================================================
if stage_runs(6):
    banner("STAGE 6  integrity check")
    require(F_MASTER, "STAGE 3")
    problems = []

    def check(cond, msg):
        print(("  ok    " if cond else "  FAIL  ") + msg)
        if not cond:
            problems.append(msg)

    M = pd.read_csv(F_MASTER)
    n = CFG.evaluation_count

    print("\nsample and master file")
    if os.path.exists(F_SELECTED):
        sel = pd.read_csv(F_SELECTED)
        check(len(sel) == CFG.demonstration_count + n, f"{len(sel)} rows selected")
        check((sel.role == "demonstration").sum() == CFG.demonstration_count, "2 demonstrations")
        check((sel.role == "evaluation").sum() == n, f"{n} evaluation cases")
        check(sel.source_row_index.nunique() == len(sel), "source rows unique")
        d = set(sel[sel.role == "demonstration"].source_row_index)
        e = set(sel[sel.role == "evaluation"].source_row_index)
        check(d.isdisjoint(e), "no demonstration leakage into the evaluation set")
    else:
        print("  (selected_cases.csv not in this session; skipping sample checks)")

    check(len(M) == n * len(CONDITIONS), f"{len(M)} master rows")
    check(M.source_row_index.nunique() == n, f"{M.source_row_index.nunique()} cases")
    check(M.groupby("condition").size().eq(n).all(), "equal rows per condition")
    check(M.groupby("source_row_index").condition.nunique().eq(len(CONDITIONS)).all(),
          "three conditions per case")
    check(M.groupby("source_row_index").reference.nunique().eq(1).all(), "one reference per case")
    check(not M.duplicated(["source_row_index", "condition"]).any(), "unique case-condition keys")
    check(M[METRICS].notna().all().all(), "no missing metric values")
    check(np.isfinite(M[METRICS].to_numpy()).all(), "all metric values finite")
    check(M[["rouge1", "rouge2", "rougeL"]].apply(lambda c: c.between(0, 1).all()).all(),
          "ROUGE in [0,1]")
    check(M.bleu.between(0, 100).all(), "BLEU in [0,100]")
    for tag, _ in JUDGES:
        c = f"{tag}_mean"
        if c in M:
            v = M[c].dropna()
            check(v.between(1, 5).all(), f"{c} in [1,5]")
            comp = M[[f"{tag}_{d}" for d in DIMS3]].mean(axis=1)
            ok = M[c].notna()
            check(np.allclose(comp[ok], M.loc[ok, c]), f"{c} equals mean of its three dimensions")
            floor = 0.99 if tag == "judge14" else 0.95
            check(v.notna().sum() / len(M) >= floor,
                  f"{tag} parsed for at least {floor:.0%} of rows "
                  f"({v.notna().sum()}/{len(M)})")

    # ---- headline numbers recomputed from scratch, sharing no helper code ----
    print("\nindependent recomputation of the headline figures")
    dfl = M.copy()
    for m in METRICS:
        sub = dfl[[m, PRIMARY]].dropna()
        a = spearmanr(sub[m], sub[PRIMARY])[0]
        b = sp(dfl[m], dfl[PRIMARY])
        check(abs(a - b) < 1e-10, f"pooled {m}: {a:+.4f}")

        # partial correlation rebuilt by hand from ranks and tier means
        s2 = dfl[[m, PRIMARY, "condition"]].dropna().copy()
        s2["xr"] = s2[m].rank()
        s2["yr"] = s2[PRIMARY].rank()
        s2["xres"] = s2.xr - s2.groupby("condition").xr.transform("mean")
        s2["yres"] = s2.yr - s2.groupby("condition").yr.transform("mean")
        manual = spearmanr(s2.xres, s2.yres)[0]
        lib = partial_spearman(dfl, m, PRIMARY, ["condition"])
        check(abs(manual - lib) < 1e-8, f"adjusted {m}: {manual:+.4f} (hand-computed = library)")

    if os.path.exists(F_SUMMEVAL):
        V = pd.read_csv(F_SUMMEVAL)
        if "judge14" in V:
            s = V[["judge14", "human_overall"]].dropna()
            print(f"\n  SummEval judge vs human, dataset level : {spearmanr(s.judge14, s.human_overall)[0]:+.4f}")
            per = [summary_level_rho(V, "judge14", d)[0]
                   for d in V.columns if d.startswith("human_") and d != "human_overall"]
            print(f"  SummEval judge, summary level (G-Eval)  : {np.mean(per):+.4f}  "
                  f"(G-Eval GPT-4 reports 0.514 on the same statistic)")

    # ---- environment ----
    with open(F_ENV, "w") as f:
        json.dump(capture_environment(), f, indent=2)
    write_provenance(6)

    prov_path = os.path.join(WORK, "provenance.json")
    if os.path.exists(prov_path):
        prov = json.load(open(prov_path))
        print("\nper-stage provenance recorded for: " + ", ".join(sorted(prov)))
        gfp = {k: v.get("gen_fingerprint") for k, v in prov.items()}
        check(len(set(gfp.values())) == 1,
              f"one generation fingerprint across all stages ({set(gfp.values())})")
        jstages = {k: v.get("judge_fingerprint") for k, v in prov.items()
                   if k in ("stage_3", "stage_4")}
        if len(jstages) == 2:
            check(len(set(jstages.values())) == 1,
                  "clinical and SummEval judging used the same rubric and judge settings")
    else:
        print("\nNOTE: no provenance.json (stage 1 predates this check)")
    with open(F_REQS, "w") as f:
        f.write(subprocess.run("pip freeze", shell=True, capture_output=True,
                               text=True).stdout)
    # keep a copy of the source that produced these results, so the manifest can
    # hash the code as well as the data
    try:
        src = os.path.abspath(sys.argv[0]) if sys.argv and sys.argv[0].endswith(".py") else None
        if src and os.path.exists(src):
            shutil.copy(src, os.path.join(WORK, "thesis_pipeline.py"))
    except Exception:                                            # noqa: BLE001
        pass

    # ---- manifest ----
    def sha256(path):
        h = hashlib.sha256()
        with open(path, "rb") as fh:
            for blk in iter(lambda: fh.read(1 << 20), b""):
                h.update(blk)
        return h.hexdigest()

    # Anything carrying MIMIC-derived text is RESTRICTED, not just the master file.
    # Under-labelling here is a data-use-agreement problem, not a tidiness problem.
    RESTRICTED = {
        os.path.basename(F_SELECTED): "seeded sample and demonstration split",
        os.path.basename(F_GEN): "generated summaries, long format",
        os.path.basename(F_METRICS): "summaries plus automatic metrics",
        os.path.basename(F_MASTER): "authoritative 900-row master",
        os.path.basename(F_JUDGE_RAW): "raw clinical judge outputs, unparsed",
        "gen_checkpoint.jsonl": "generation checkpoint, contains summary text",
        "judge_audit_sheet.csv": "manual audit sample, contains summary text",
    }
    SHAREABLE = {
        os.path.basename(F_SUMMEVAL): "SummEval validation (public dataset)",
        os.path.basename(F_SUMM_RAW): "raw SummEval judge outputs (public dataset)",
        os.path.basename(F_CONFIG): "frozen experiment configuration",
        os.path.basename(F_ENV): "software environment",
        os.path.basename(F_TRUNC): "input truncation report (counts only)",
        "provenance.json": "per-stage environment and fingerprints",
        "model_revisions.json": "judge model commit SHAs",
        "requirements-final.txt": "pinned package versions",
        "thesis_pipeline.py": "pipeline source",
    }

    def count_rows(path, fn):
        try:
            # clinical text contains embedded newlines, so line counting lies
            return len(pd.read_csv(path)) if fn.endswith(".csv") else sum(1 for _ in open(path))
        except Exception:                                        # noqa: BLE001
            return -1

    entries = []
    for cls, group in (("RESTRICTED", RESTRICTED), ("SHAREABLE", SHAREABLE)):
        for fn, why in group.items():
            fp = os.path.join(WORK, fn)
            if os.path.exists(fp):
                entries.append((cls, fn, count_rows(fp, fn), sha256(fp), why))
    # every generated table, figure and bootstrap file
    for d in (OUTPUTS, os.path.join(WORK, "public_results")):
        if not os.path.isdir(d):
            continue
        for fn in sorted(os.listdir(d)):
            fp = os.path.join(d, fn)
            if os.path.isfile(fp):
                entries.append(("SHAREABLE", os.path.relpath(fp, WORK),
                                count_rows(fp, fn), sha256(fp), "generated output"))

    lines = [f"{'class':12s}{'file':46s}{'rows':>8s}  {'sha256':64s}  purpose"]
    for cls, fn, nrows, h, why in entries:
        lines.append(f"{cls:12s}{fn:46s}{nrows:8d}  {h:64s}  {why}")
    lines.append(f"\ngenerated {time.strftime('%Y-%m-%d %H:%M:%S')}")
    lines.append(f"run_label={CFG.run_label}  gen_fingerprint={fingerprint(GEN_KEYS)}")
    lines.append("RESTRICTED files contain MIMIC-IV derived text. They must stay inside "
                 "your credentialed environment and must not be attached to the thesis, "
                 "pushed to a repository, or shared with anyone lacking PhysioNet access.")
    with open(F_MANIFEST, "w") as f:
        f.write("\n".join(lines) + "\n")
    print("\n" + "\n".join(lines))

    print("\n" + "=" * 78)
    if problems:
        print(f"{len(problems)} CHECK(S) FAILED:")
        for p in problems:
            print("  -", p)
        raise SystemExit(1)
    print("ALL INTEGRITY CHECKS PASSED")
    print("=" * 78)